In [0]:
%python
# Create Catalog & Schemas
spark.sql("CREATE CATALOG IF NOT EXISTS stock_analytics")
spark.sql("USE CATALOG stock_analytics")

schemas = ["bronze", "silver", "gold"]
for schema in schemas:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS stock_analytics.{schema}")
    print(f"✅ Schema created: stock_analytics.{schema}")

In [0]:
%python
# Create Volume
spark.sql("""
    CREATE VOLUME IF NOT EXISTS stock_analytics.bronze.raw
""")
print("✅ Volume created: stock_analytics.bronze.raw")

In [0]:
# Create folder structure
folders = [
    "/Volumes/stock_analytics/bronze/raw/daily/",
    "/Volumes/stock_analytics/bronze/raw/schema/"
]

for folder in folders:
    dbutils.fs.mkdirs(folder)
    print(f"✅ Created: {folder}")

print("\n✅ Volume folder structure ready!")

In [0]:
# Create Gold Delta tables
spark.sql("""
    CREATE TABLE IF NOT EXISTS stock_analytics.gold.daily_prices (
        ticker           STRING,
        trade_date       DATE,
        open             DOUBLE,
        high             DOUBLE,
        low              DOUBLE,
        close            DOUBLE,
        volume           DOUBLE,
        daily_return_pct DOUBLE,
        price_range      DOUBLE,
        moving_avg_7d    DOUBLE,
        moving_avg_30d   DOUBLE,
        data_source      STRING,
        ingested_at      TIMESTAMP
    )
    USING DELTA
    COMMENT 'Daily OHLCV prices for top 10 S&P 500 stocks'
""")
print("✅ Table created: stock_analytics.gold.daily_prices")

spark.sql("""
    CREATE TABLE IF NOT EXISTS stock_analytics.gold.stock_volatility (
        ticker             STRING,
        volatility_stddev  DOUBLE,
        avg_daily_return   DOUBLE,
        best_day_return    DOUBLE,
        worst_day_return   DOUBLE,
        total_trading_days LONG,
        risk_tier          STRING,
        data_from          DATE,
        data_to            DATE
    )
    USING DELTA
    COMMENT 'Volatility metrics and risk tier per stock'
""")
print("✅ Table created: stock_analytics.gold.stock_volatility")

spark.sql("""
    CREATE TABLE IF NOT EXISTS stock_analytics.gold.monthly_performance (
        ticker             STRING,
        month              STRING,
        monthly_return_pct DOUBLE,
        best_single_day    DOUBLE,
        worst_single_day   DOUBLE,
        avg_close          DOUBLE,
        trading_days       LONG,
        rank               INT
    )
    USING DELTA
    COMMENT 'Monthly return rankings per stock'
""")
print("✅ Table created: stock_analytics.gold.monthly_performance")

In [0]:
# Verify full structure
print("=" * 50)

print("\n📋 SCHEMAS:")
spark.sql("SHOW SCHEMAS IN stock_analytics").show()

print("📋 VOLUME:")
spark.sql("SHOW VOLUMES IN stock_analytics.bronze").show()

print("📋 GOLD TABLES:")
spark.sql("SHOW TABLES IN stock_analytics.gold").show()

print("📋 FOLDERS IN VOLUME:")
display(dbutils.fs.ls("/Volumes/stock_analytics/bronze/raw/"))

print("\n✅ Setup complete — ready for 01_data_loader.py!")